## Tutorial

Release

:   6.1.2

Date

:   Oct 28, 2026

This guide can help you start working with FLPQ_Data.

**You can download this tutorial as a Jupyter Notebook from the link at the end of the page.**

**NetworkX.** We use the NetworkX [MultiDiGraph](https://networkx.org/documentation/latest/reference/classes/multidigraph.html) to represent the labeled graph.
To familiarize yourself with this representation and find useful functions, see [NetworkX tutorial](https://networkx.org/documentation/latest/tutorial.html).

**Pyformlang.** We use the Pyformlang [Regex](https://pyformlang.readthedocs.io/en/latest/modules/regular_expression.html) to represent regular grammars.
Also, we use the Pyformlang [CFG](https://pyformlang.readthedocs.io/en/latest/modules/context_free_grammar.html) and [RSA](https://pyformlang.readthedocs.io/en/latest/modules/rsa.html) to represent context-free grammars.
To familiarize yourself with this representations and find useful functions, see [Pyformlang usage](https://pyformlang.readthedocs.io/en/latest/usage.html).

All functions are documented on the Reference page.

### Import

First you need to import the package.

In [1]:
import flpq_data

# Load graph

After the package is imported, we can load the graphs.

## Load graph archive from Dataset

We can load the archive with the graph using function `graph_dir`: it returns the local directory of the graph and downloads it into the machine-global cache only when it is not there yet.

In [2]:
bzip_path = flpq_data.graph_dir("bzip")

[2026-10-28 10:13:55]>INFO>Cache hit for graph name='bzip'


# Load graph by path

The archive unpacks to a directory with one MatrixMarket file per edge
label in its `graph` subdirectory (see File structure). We
can load the graph along the specified path using function
`graph_from_mtx_dir`.

In [3]:
bzip = flpq_data.graph_from_mtx_dir(bzip_path / "graph")

[2026-10-28 10:13:55]>INFO>Map filename='a.mtx' to label='a'


[2026-10-28 10:13:55]>INFO>Map filename='d.mtx' to label='d'


[2026-10-28 10:13:55]>INFO>Stream edges from path=PosixPath('/home/runner/.cache/flpq-data/6.0.0/graph/bzip/graph')


[2026-10-28 10:13:55]>INFO>Load graph=<networkx.classes.multidigraph.MultiDiGraph object at 0x7f714008f610> from path=PosixPath('/home/runner/.cache/flpq-data/6.0.0/graph/bzip/graph')


Create graph

We can also create a synthetic graph using one of the generators in module Graph generators.

# Create a one cycle graph

For example, let’s create a one cycle graph, with 5 nodes, the edges of which are marked with the letter `a`.

In [4]:
cycle = flpq_data.labeled_cycle_graph(5, label="a")

[2026-10-28 10:13:55]>INFO>Create a cycle graph=<networkx.classes.multidigraph.MultiDiGraph object at 0x7f7134592050> with n=5, label='a'


Change edges

We can change the specified graph labels by using function `change_edges`
from Graph utilities.

In [5]:
new_cycle = flpq_data.change_edges(cycle, {"a": "b"})

[2026-10-28 10:13:55]>INFO>Change labels in graph=<networkx.classes.multidigraph.MultiDiGraph object at 0x7f7134592050> with mapping={'a': 'b'} to new_graph=<networkx.classes.multidigraph.MultiDiGraph object at 0x7f71400ae910>


Now the labels `a` have changed to `b`.

# Add reverse edges

In addition, we can add reverse edges to the graph by using function `add_reverse_edges`
from Graph utilities. This is extremely useful if graph analysis is formulated using such reverse edges.

In [6]:
new_cycle_with_reversed = flpq_data.add_reverse_edges(new_cycle)

[2026-10-28 10:13:55]>INFO>Add reverse edges in graph=<networkx.classes.multidigraph.MultiDiGraph object at 0x7f71400ae910> with mapping=None to new_graph=<networkx.classes.multidigraph.MultiDiGraph object at 0x7f712014f4d0>


Now, for each edge with label `a` this graph contains the reversed edge with label `a_r`.

# Convert graph format

Graphs can be converted between the supported formats — `mtx` (a directory
of MatrixMarket files), `txt`, `rdf`, and the FastMatrixCFPQ `.g` text
— with function `convert_graph`:
the conversion runs over a streaming edge iterator without building an
in-memory graph, so even large archives convert in constant memory (see the
Format conversion section).

In [7]:
txt_path = flpq_data.convert_graph(
    bzip_path / "graph", "bzip.txt", src_format="mtx", dst_format="txt"
)

[2026-10-28 10:13:55]>INFO>Map filename='a.mtx' to label='a'


[2026-10-28 10:13:55]>INFO>Map filename='d.mtx' to label='d'


[2026-10-28 10:13:55]>INFO>Stream edges from path=PosixPath('/home/runner/.cache/flpq-data/6.0.0/graph/bzip/graph')


[2026-10-28 10:13:55]>INFO>Save edges to dest=PosixPath('/home/runner/work/FLPQ_Data/FLPQ_Data/docs/bzip.txt')


[2026-10-28 10:13:55]>INFO>Convert src=PosixPath('/home/runner/.cache/flpq-data/6.0.0/graph/bzip/graph') from src_format='mtx' to dst_format='txt'


# Load grammar

Graph archives from the dataset are self-contained: besides the graph they carry the
queries that apply to the graph under `queries/` — one directory per query with its
grammar template and a precomputed `results.mtx` (see File structure).
The grammar templates themselves are described in the CFPQ section.

A grammar template may use indexed symbols (e.g. `load_i`); we materialize it over a
concrete graph with functions `cnf_template_from_text`
and `materialize`,
which expand every index present in the graph edge labels:

In [8]:
import networkx as nx
g = nx.MultiDiGraph()
_ = g.add_edges_from(
    [(0, 1, {"label": "load_0"}), (1, 2, {"label": "store_0"}),
     (0, 3, {"label": "alloc"})]
)
text = ("PT\tPTh\talloc\n"
        "PT\talloc\n"
        "PTh\tload_i\tAl_st_PTh_i\n"
        "Al_st_PTh_i\tAl\tst_PTh_i\n"
        "st_PTh_i\tstore_i\tPTh\n"
        "Al\tPT\n"
        "\n"
        "Count:\n"
        "PT")
template = flpq_data.cnf_template_from_text(text)
cfg = flpq_data.materialize(template, g)

[2026-10-28 10:13:55]>INFO>Create cfg=<pyformlang.cfg.cfg.CFG object at 0x7f712015b110> from text='PT\tPTh\talloc\nPT\talloc\nPTh\tload_i\tAl_st_PTh_i\nAl_st_PTh_i\tAl\tst_PTh_i\nst_PTh_i\tstore_i\tPTh\nAl\tPT\n\nCount:\nPT'


[2026-10-28 10:13:55]>INFO>Materialize cfg=<pyformlang.cfg.cfg.CFG object at 0x7f712015b110> over graph=<networkx.classes.multidigraph.MultiDiGraph object at 0x7f712015a1d0> to result=<pyformlang.cfg.cfg.CFG object at 0x7f712015bf90>


# Regular grammars

Currently, we have one representation of regular grammars:

1. [Regular expression](https://en.wikipedia.org/wiki/Regular_expression#Formal_definition)

## Create a regular expression

For example, a regular expression can be created by using function `regex_from_text`
from Reading and writing grammars.

In [9]:
regex = flpq_data.regex_from_text("a (bc|d*)")

[2026-10-28 10:13:56]>INFO>Create regex=(a.(bc|(d)*)) from text='a (bc|d*)'


# Load regular expression by path

We can load the regular expression along the specified path using function `regex_from_txt`.

In [10]:
path = flpq_data.regex_to_txt(regex, "test.txt")
regex_by_path = flpq_data.regex_from_txt(path)

[2026-10-28 10:13:56]>INFO>Turn regex=(a.(bc|(d)*)) into text='(a (bc|(d)*))'


[2026-10-28 10:13:56]>INFO>Save regex=(a.(bc|(d)*)) to dest=PosixPath('/home/runner/work/FLPQ_Data/FLPQ_Data/docs/test.txt')


[2026-10-28 10:13:56]>INFO>Create regex=(a.(bc|(d)*)) from text='(a (bc|(d)*))'


[2026-10-28 10:13:56]>INFO>Create regex=(a.(bc|(d)*)) from path=PosixPath('/home/runner/work/FLPQ_Data/FLPQ_Data/docs/test.txt')


Сontext-free grammars

Currently, we have three representations of context-free grammars (CFGs):

1. [Classic](https://en.wikipedia.org/wiki/Context-free_grammar#Formal_definitions)

1. [Chomsky Normal Form](https://en.wikipedia.org/wiki/Chomsky_normal_form)

1. [Recursive State Machine](https://link.springer.com/chapter/10.1007/978-3-030-54832-2_6)

# Create a classic context-free grammar

A classic context-free grammar can be created by using function `cfg_from_text`
from Reading and writing grammars.

In [11]:
cfg = flpq_data.cfg_from_text("S -> a S b S | a b")

[2026-10-28 10:13:56]>INFO>Create cfg=<pyformlang.cfg.cfg.CFG object at 0x7f712015b510> from text='S -> a S b S | a b', start_symbol=Variable(S)


# Load context-free grammar by path

We can load the classic context-free grammar along the specified path using function `cfg_from_txt`.

In [12]:
path = flpq_data.cfg_to_txt(cfg, "test.txt")
cfg_by_path = flpq_data.cfg_from_txt(path)

[2026-10-28 10:13:56]>INFO>Turn cfg=<pyformlang.cfg.cfg.CFG object at 0x7f712015b510> into text='S -> a S b S\nS -> a b'


[2026-10-28 10:13:56]>INFO>Save cfg=<pyformlang.cfg.cfg.CFG object at 0x7f712015b510> to dest=PosixPath('/home/runner/work/FLPQ_Data/FLPQ_Data/docs/test.txt')


[2026-10-28 10:13:56]>INFO>Create cfg=<pyformlang.cfg.cfg.CFG object at 0x7f712015be50> from text='S -> a S b S\nS -> a b', start_symbol=Variable(S)


[2026-10-28 10:13:56]>INFO>Create cfg=<pyformlang.cfg.cfg.CFG object at 0x7f712015be50> from path=PosixPath('/home/runner/work/FLPQ_Data/FLPQ_Data/docs/test.txt'), start_symbol=Variable(S)


Generate grammar

We can also generate a grammar for specified template using one of the generators in module Grammar generators.

# Generate a Dyck grammar

For example, let’s generate a Dyck grammar of the balanced strings with `a` as an opening parenthesis, `b` as a closing parenthesis, and without the empty string.

In [13]:
dyck_cfg = flpq_data.dyck_grammar([("a", "b")], eps=False)

[2026-10-28 10:13:56]>INFO>Create a Dyck cfg=<pyformlang.cfg.cfg.CFG object at 0x7f7120160250> with types=[('a', 'b')], eps=False


# Generate a Java Points-to grammar

Also, let’s generate a Java Points-to grammar for the field-sensitive analysis of Java programs with field names `f0` and `f1`.

In [14]:
java_pt_cfg = flpq_data.java_points_to_grammar(["f0", "f1"])

[2026-10-28 10:13:56]>INFO>Create a Java Points-to cfg=<pyformlang.cfg.cfg.CFG object at 0x7f7120161350> with fields=['f0', 'f1']
